# Module 03 — Lesson 04
# Basic Probability

---

> "70% chance of rain tomorrow." You've heard this a thousand times — but what does it actually mean?

It does NOT mean "it will rain for 70% of the day." It means: out of all the past days with weather conditions like today's, it rained on 70% of them. **Probability is a number between 0 and 1** (or 0% and 100%) that measures how likely an event is.

Probability is the mathematical language of uncertainty, and it sits underneath almost every ML model:
- A spam filter doesn't say "this IS spam" — it outputs P(spam | email content) = 0.94
- A recommendation system ranks movies by predicted probability you'll like them
- An A/B test asks: "what's the probability this improvement is due to random chance?"
- A medical diagnosis model outputs P(disease | symptoms), not a plain yes/no

By the end of this lesson, you'll compute probabilities like these from first principles.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Define probability and compute it using the classical (theoretical) definition
- Distinguish theoretical vs experimental (empirical) probability, and see the Law of Large Numbers in action
- Apply the Addition Rule to find P(A or B), with and without overlap
- Apply the Multiplication Rule to find P(A and B), for independent and dependent events
- Compute conditional probability P(A|B) and avoid the base-rate fallacy
- Apply Bayes' Theorem to update a belief given new evidence

---
## 1. What Is Probability?

The **classical (theoretical) definition** of probability, for outcomes that are all equally likely:

$$P(\text{event}) = \frac{\text{number of favorable outcomes}}{\text{total number of possible outcomes}}$$

Probability always falls between 0 (impossible) and 1 (certain). The full set of possible outcomes is called the **sample space**.

In [ ]:
# Sample space: all possible outcomes of rolling a single six-sided die
sample_space = [1, 2, 3, 4, 5, 6]

def probability(favorable, sample_space):
    '''Classical probability: favorable outcomes / total outcomes.'''
    return len(favorable) / len(sample_space)

# P(rolling a 4)
p_four = probability([4], sample_space)

# P(rolling an even number)
evens = [x for x in sample_space if x % 2 == 0]
p_even = probability(evens, sample_space)

# P(rolling a number greater than 4)
gt_four = [x for x in sample_space if x > 4]
p_gt_four = probability(gt_four, sample_space)

print(f"Sample space         : {sample_space}")
print(f"P(rolling a 4)        : {p_four:.3f}  ({p_four*100:.1f}%)")
print(f"P(rolling even)       : {p_even:.3f}  ({p_even*100:.1f}%)")
print(f"P(rolling > 4)        : {p_gt_four:.3f}  ({p_gt_four*100:.1f}%)")
print()
print("Probability is always between 0 (impossible) and 1 (certain).")

---
## 2. Sample Space and Events — Compound Outcomes

In [ ]:
# Sample space for rolling TWO dice: all 36 (die1, die2) combinations
two_dice_space = [(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)]

# Event: the two dice sum to 7
sum_to_7 = [(d1, d2) for d1, d2 in two_dice_space if d1 + d2 == 7]

# Event: both dice show the same number (a "double")
doubles = [(d1, d2) for d1, d2 in two_dice_space if d1 == d2]

p_sum_7 = probability(sum_to_7, two_dice_space)
p_double = probability(doubles, two_dice_space)

print(f"Total outcomes (2 dice) : {len(two_dice_space)}")
print(f"Outcomes summing to 7   : {sum_to_7}")
print(f"P(sum == 7)   : {p_sum_7:.3f}  ({p_sum_7*100:.1f}%)")
print()
print(f"Outcomes that are doubles : {doubles}")
print(f"P(double)     : {p_double:.3f}  ({p_double*100:.1f}%)")

---
## 3. Theoretical vs Experimental Probability

**Theoretical probability** is calculated in advance from the sample space (what we just did). **Experimental (empirical) probability** is measured by actually running trials:

$$P_{\text{experimental}} = \frac{\text{number of times event occurred}}{\text{number of trials}}$$

The **Law of Large Numbers** says: as the number of trials grows, experimental probability converges toward theoretical probability.

In [ ]:
import random
random.seed(42)

def flip_coin():
    return random.choice(["H", "T"])

trial_counts = [10, 100, 1000, 10000, 100000]

print(f"{'Flips':>8} {'Heads':>8} {'P(Heads) experimental':>24} {'Theoretical':>12}")
print("-" * 60)
for n in trial_counts:
    flips = [flip_coin() for _ in range(n)]
    heads = flips.count("H")
    p_exp = heads / n
    print(f"{n:>8} {heads:>8} {p_exp:>23.4f}  {0.5:>11.4f}")

print()
print("As the number of flips grows, the experimental probability converges")
print("toward the theoretical probability of 0.5. This is the LAW OF LARGE NUMBERS.")

---
## 4. The Addition Rule — P(A or B)

$$P(A \text{ or } B) = P(A) + P(B) - P(A \text{ and } B)$$

The subtraction removes double-counting when A and B can both happen at once. If the events are **mutually exclusive** (can't both happen), $P(A \text{ and } B) = 0$ and the rule simplifies to a plain sum.

In [ ]:
# A standard deck: 52 cards, 13 ranks x 4 suits
ranks = ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]
suits = ["Hearts","Diamonds","Clubs","Spades"]
deck = [(r, s) for s in suits for r in ranks]

kings = [c for c in deck if c[0] == "K"]
hearts = [c for c in deck if c[1] == "Hearts"]
king_of_hearts = [c for c in deck if c[0] == "K" and c[1] == "Hearts"]

p_king = probability(kings, deck)
p_heart = probability(hearts, deck)
p_king_and_heart = probability(king_of_hearts, deck)

# Addition rule (NOT mutually exclusive — a King of Hearts is both!)
p_king_or_heart = p_king + p_heart - p_king_and_heart

print(f"P(King)             : {p_king:.4f}  ({len(kings)}/52)")
print(f"P(Heart)             : {p_heart:.4f}  ({len(hearts)}/52)")
print(f"P(King AND Heart)    : {p_king_and_heart:.4f}  ({len(king_of_hearts)}/52)")
print(f"P(King OR Heart)     : {p_king_or_heart:.4f}")
print()

# Mutually exclusive example: King or Queen (a card can't be both)
queens = [c for c in deck if c[0] == "Q"]
p_queen = probability(queens, deck)
p_king_or_queen = p_king + p_queen   # no overlap, no subtraction needed
print(f"P(King OR Queen)     : {p_king_or_queen:.4f}  (mutually exclusive — simple sum)")

---
## 5. The Multiplication Rule — P(A and B)

For **independent** events (one doesn't affect the other):

$$P(A \text{ and } B) = P(A) \times P(B)$$

In [ ]:
p_heads = 0.5
p_two_heads = p_heads * p_heads

p_six = 1 / 6
p_two_sixes = p_six * p_six

print(f"P(heads on flip 1 AND heads on flip 2) : {p_two_heads:.4f}  ({p_heads} x {p_heads})")
print(f"P(rolling a 6 twice in a row)           : {p_two_sixes:.4f}  ({p_six:.3f} x {p_six:.3f})")
print()
print("The multiplication rule applies because each flip/roll does NOT affect")
print("the next one — the events are INDEPENDENT.")

---
## 6. Independent vs Dependent Events

In [ ]:
# Independent: drawing a card, replacing it, then drawing again
p_king_first_indep = 4 / 52
p_king_second_indep = 4 / 52   # deck reset — same probability
p_both_kings_indep = p_king_first_indep * p_king_second_indep

# Dependent: drawing two cards WITHOUT replacement
p_king_first_dep = 4 / 52
p_king_second_dep = 3 / 51     # one king removed, one card removed from deck
p_both_kings_dep = p_king_first_dep * p_king_second_dep

print("WITH replacement (independent):")
print(f"  P(King, then King) = {p_king_first_indep:.4f} x {p_king_second_indep:.4f} = {p_both_kings_indep:.4f}")
print()
print("WITHOUT replacement (dependent):")
print(f"  P(King, then King) = {p_king_first_dep:.4f} x {p_king_second_dep:.4f} = {p_both_kings_dep:.4f}")
print()
print("Removing a card changes what's left in the deck — the second draw's")
print("probability DEPENDS on the outcome of the first. This is a dependent event.")

---
## 7. Conditional Probability — P(A | B)

**Conditional probability** answers: "given that B has already happened, what's the probability of A?"

$$P(A \mid B) = \frac{P(A \text{ and } B)}{P(B)}$$

In [ ]:
# Survey of 100 students: did they study, and did they pass?
#                  Passed   Failed   Total
# Studied            42        8       50
# Didn't study       18       32       50
# Total              60       40      100

total_students = 100
studied_and_passed = 42
studied = 50
passed = 60

p_studied = studied / total_students
p_passed = passed / total_students
p_studied_and_passed = studied_and_passed / total_students

# P(Passed | Studied) — given that a student studied, probability they passed
p_passed_given_studied = p_studied_and_passed / p_studied

print(f"P(Studied)               : {p_studied:.2f}")
print(f"P(Passed)                : {p_passed:.2f}")
print(f"P(Studied AND Passed)    : {p_studied_and_passed:.2f}")
print()
print(f"P(Passed | Studied)      : {p_passed_given_studied:.2f}  <- conditional probability")
print()
print(f"Overall pass rate is {p_passed:.0%}, but AMONG students who studied,")
print(f"the pass rate jumps to {p_passed_given_studied:.0%}. Conditioning on 'studied'")
print("changes the probability — that's the whole point of conditional probability.")

---
## 8. Bayes' Theorem — Updating Beliefs With Evidence

Bayes' Theorem flips a conditional probability around:

$$P(A \mid B) = \frac{P(B \mid A) \times P(A)}{P(B)}$$

It's how you update a belief (P(A)) after observing new evidence (B). Classic example: **medical testing**.

In [ ]:
# A disease affects 1% of the population.
# A test correctly detects the disease 95% of the time (true positive rate)
# The test incorrectly flags healthy people as positive 5% of the time (false positive rate)
#
# If someone tests POSITIVE, what's the actual probability they have the disease?

p_disease = 0.01
p_no_disease = 1 - p_disease
p_positive_given_disease = 0.95        # true positive rate (sensitivity)
p_positive_given_no_disease = 0.05     # false positive rate

# P(Positive) = P(Pos|Disease)*P(Disease) + P(Pos|NoDisease)*P(NoDisease)
p_positive = (p_positive_given_disease * p_disease) + (p_positive_given_no_disease * p_no_disease)

# Bayes' Theorem: P(Disease | Positive) = P(Positive|Disease) * P(Disease) / P(Positive)
p_disease_given_positive = (p_positive_given_disease * p_disease) / p_positive

print(f"P(Disease)                     : {p_disease:.2%}")
print(f"P(Positive test | Disease)     : {p_positive_given_disease:.2%}  (test's stated accuracy)")
print(f"P(Positive test | No disease)  : {p_positive_given_no_disease:.2%}  (false positive rate)")
print()
print(f"P(Positive test)  [overall]    : {p_positive:.4f}")
print(f"P(Disease | Positive test)     : {p_disease_given_positive:.2%}  <- Bayes' Theorem result")
print()
print(f"Even with a 95%-accurate test, a positive result only means a")
print(f"{p_disease_given_positive:.0%} chance of actually having the disease — because the")
print("disease is so rare, false positives from the huge healthy population")
print("outnumber true positives from the tiny sick population.")

---
## 9. Verification via Simulation

In [ ]:
import random
random.seed(1)

def simulate_bayes(n_people=200_000):
    '''Simulate the disease/test scenario to verify the Bayes Theorem result.'''
    infected_and_positive = 0
    total_positive = 0

    for _ in range(n_people):
        has_disease = random.random() < p_disease
        if has_disease:
            tested_positive = random.random() < p_positive_given_disease
        else:
            tested_positive = random.random() < p_positive_given_no_disease

        if tested_positive:
            total_positive += 1
            if has_disease:
                infected_and_positive += 1

    return infected_and_positive / total_positive


simulated = simulate_bayes()
print(f"Bayes' Theorem (calculated) : {p_disease_given_positive:.4f}")
print(f"Monte Carlo simulation      : {simulated:.4f}")
print(f"Difference                  : {abs(simulated - p_disease_given_positive):.4f}")
print()
print("The simulation confirms the formula — simulating 200,000 people and")
print("directly counting matches what Bayes' Theorem predicts analytically.")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Confusing P(A|B) with P(B|A) — the Base Rate Fallacy ----------

print("Mistake 1 — P(A|B) is NOT the same as P(B|A):")
print(f"  P(Positive test | Disease)  = {p_positive_given_disease:.0%}   <- the test's sensitivity")
print(f"  P(Disease | Positive test)  = {p_disease_given_positive:.0%}   <- actual chance you're sick")
print()
print("  These are DIFFERENT numbers answering different questions.")
print("  'The test is 95% accurate' does NOT mean 'a positive result means")
print("  95% chance of disease.' This mix-up is called the BASE RATE FALLACY —")
print("  it shows up constantly in medical diagnosis and legal reasoning.")

In [ ]:
# -- Mistake 2: Forgetting to subtract the overlap in the Addition Rule -------

deck = [(r, s) for s in ["Hearts","Diamonds","Clubs","Spades"]
        for r in ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]]
kings = [c for c in deck if c[0] == "K"]
hearts = [c for c in deck if c[1] == "Hearts"]

wrong = len(kings) / 52 + len(hearts) / 52   # WRONG — double-counts King of Hearts
correct = wrong - (1 / 52)                   # subtract the overlap

print("Mistake 2 — Double-counting overlapping events:")
print(f"  WRONG:   P(King) + P(Heart)                      = {wrong:.4f}")
print(f"  CORRECT: P(King) + P(Heart) - P(King AND Heart)  = {correct:.4f}")
print()
print("  The King of Hearts got counted TWICE in the wrong version — once as")
print("  a King, once as a Heart. Always check whether events overlap before")
print("  using the simple addition rule.")

In [ ]:
# -- Mistake 3: Assuming independence when events are dependent --------------

# WRONG: treating "no replacement" draws as independent
p_wrong = (4 / 52) * (4 / 52)

# CORRECT: accounting for the changed deck after the first draw
p_correct = (4 / 52) * (3 / 51)

print("Mistake 3 — Wrongly assuming independence:")
print(f"  WRONG (assumes replacement)   : P(two Kings) = {p_wrong:.4f}")
print(f"  CORRECT (no replacement)      : P(two Kings) = {p_correct:.4f}")
print()
print("  Always ask: 'does the first event change the odds of the second?'")
print("  Sampling without replacement is dependent. Sampling WITH replacement")
print("  (or independent physical events like coin flips) is not.")

In [ ]:
# -- Mistake 4: Trusting a small sample's experimental probability -----------

import random
random.seed(7)

small_sample = [random.choice(["H", "T"]) for _ in range(10)]
p_small = small_sample.count("H") / 10

print("Mistake 4 — Small samples are noisy:")
print(f"  10 coin flips: {small_sample}")
print(f"  Experimental P(Heads) from 10 flips: {p_small:.2f}")
print()
print("  A fair coin has a TRUE probability of exactly 0.5 — but with only 10")
print("  flips, the experimental result can easily be 0.3 or 0.7 purely by chance.")
print("  Don't trust probability estimates from small samples. More data =")
print("  more reliable estimate (Law of Large Numbers, Section 3).")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Two-Dice Probability

Using the `two_dice_space` idea from Section 2, compute the probability that the sum of two dice is **10 or greater**.

```python
two_dice_space = [(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)]
```

In [ ]:
two_dice_space = [(d1, d2) for d1 in range(1, 7) for d2 in range(1, 7)]
# Your code here

### Exercise 2 — Card Addition Rule

Using the `deck` from Section 4, compute `P(Ace or Spade)`. Are these events mutually exclusive? Show your work with the Addition Rule.

In [ ]:
deck = [(r, s) for s in ["Hearts","Diamonds","Clubs","Spades"]
        for r in ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]]
# Your code here

### Exercise 3 — Coin Flip Convergence

Write `experimental_probability(n_flips)` that simulates `n_flips` coin tosses and returns the proportion of heads. Call it for `n_flips` in `[5, 50, 500, 5000, 50000]` and print how the result gets closer to 0.5 as `n_flips` grows.

In [ ]:
import random

def experimental_probability(n_flips):
    # Your code here
    pass

for n in [5, 50, 500, 5000, 50000]:
    print(n, experimental_probability(n))

### Exercise 4 — Conditional Probability: Umbrellas and Rain

A survey of 200 commuters recorded whether they carried an umbrella and whether it rained that day:

|             | Rained | Didn't Rain | Total |
|---|---|---|---|
| Carried umbrella    | 54 | 26 | 80 |
| No umbrella         | 16 | 104 | 120 |
| **Total**           | 70 | 130 | 200 |

Compute `P(Rained | Carried umbrella)` and `P(Carried umbrella | Rained)`. Are they the same number? Explain why or why not.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Spam Filter with Bayes' Theorem

A spam filter has these known rates:
- `P(spam)` = 0.20 (20% of all incoming email is spam)
- `P("free" appears | spam)` = 0.60
- `P("free" appears | not spam)` = 0.05

Using Bayes' Theorem, write `p_spam_given_word(p_spam, p_word_given_spam, p_word_given_not_spam)` that returns `P(spam | "free" appears)`. Then verify your formula with a Monte Carlo simulation of 200,000 emails, similar to Section 9.

In [ ]:
p_spam = 0.20
p_word_given_spam = 0.60
p_word_given_not_spam = 0.05

def p_spam_given_word(p_spam, p_word_given_spam, p_word_given_not_spam):
    # Your code here
    pass

print(p_spam_given_word(p_spam, p_word_given_spam, p_word_given_not_spam))

---
## 📌 Key Takeaways

- **Probability is a number between 0 and 1 that measures likelihood, computed as favorable outcomes over total outcomes** (for equally-likely outcomes). Experimental probability, measured by running trials, converges to the theoretical value as the number of trials grows — the Law of Large Numbers.

- **The Addition and Multiplication Rules combine simple events into compound ones.** Use `P(A) + P(B) - P(A and B)` for "or," subtracting the overlap unless events are mutually exclusive. Use `P(A) × P(B)` for "and" only when events are independent — check for dependence (like sampling without replacement) before multiplying.

- **Conditional probability and Bayes' Theorem let you update beliefs given evidence — and P(A|B) is never automatically equal to P(B|A).** This distinction (the base rate fallacy) is the single most common probability mistake in the real world, from medical test results to spam filters to legal reasoning. Bayes' Theorem is also the mathematical foundation of the Naive Bayes classifier you'll meet in the Classification module.

---
## What's Next?

**Lesson 05 — Random Variables and Distributions**
You now know how to compute the probability of a single event. Next, you'll learn to describe the ENTIRE range of possible outcomes and their probabilities at once — a random variable — and see how patterns like the binomial distribution model everyday scenarios like "how many of these 10 emails are spam?"